# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Xindi Liu

**ID**: xl748

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/Desktop/Cornell_Courses/FA_25/BEE_4750/hw5-xindicliu`
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed Measures ─────────── v0.3.3
   Installed GR_jll ───────────── v0.73.18+0
   Installed PlotUtils ────────── v1.4.4
   Installed OpenSSL ──────────── v1.6.0
   Installed StaticArrays ─────── v1.9.15
   Installed MutableArithmetics ─ v1.6.7
   Installed JSON ─────────────── v1.3.0
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed StaticArraysCore ─── v1.4.4
   Installed GR ───────────────── v0.73.18
   Installed Pango_jll ────────── v1.57.0+0
   Installed FFMPEG ───────────── v0.4.5
   Installed METIS_jll ────────── v5.1.3+0
   Installed DataStructures ───── v0.19.3
   Installed GraphRecipes ─────── v0.5.15
   Installed StatsBase ────────── v0.34.8
   Installed JuMP ─────────────── v1.29.3
   Installed StableRNGs ───────── v1.0.4
   Installed Adapt ────────────── v4.4.0
   Installed HiGHS ────────────── v1.20.1
   Installed FFMPEG_jll ───────── v

In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

**Answer**:

| Quantity | Formula | Actual Calculation | Value |
|---------|----------|--------------------|--------|
| Overall recycling fraction | $r = \sum w_i r_i$ | $0.15(0) + 0.40(0.55) + \ldots + 0.03(0)$ | $0.3775\;(37.75\%)$ |
| Overall MSW ash fraction | $a_{\text{MSW}} = \sum w_i a_i$ | $0.15(0.08) + 0.40(0.07) + \ldots + 0.03(0.70)$ | $0.1641\;(16.41\%)$ |

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

**Answer**:

Let:

$ x_{i,j} = $ mass flow (Mg/day) from city $i$ to facility $j$, where $i\in\{1,2,3\}$ and $j\in\{LF,MRF,WTE\}$.

$ y_{MRF,k} = $ mass flow (Mg/day) of residue from MRF to facility $k$, where $k\in\{LF,WTE\}$.

$ y_{WTE,LF} = $ mass flow (Mg/day) of ash from WTE to LF.

$ \delta_j = $ binary variable indicating whether facility $j$ is used, $j\in\{LF,MRF,WTE\}$, with $\delta_j\in\{0,1\}$.

All mass-flow variables are nonnegative.

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

**Answer**:

Parameter definitiosn: 
| Parameter | Meaning |
|----------|----------|
| $t$ | transport cost (\$/Mg·km) |
| $d_{i,j}$ | distance from city $i$ to facility $j$ |
| $c^{\text{tip}}_j$ | tipping fee at facility $j$ |
| $c^{\text{rec}}$ | recycling processing cost (\$/Mg recycled) |
| $r_{\text{overall}}$ | overall recycling fraction (=0.3775) |
| $r_{\text{MRF}}$ | MRF recycling rate (=0.40) |
| $F_j$ | fixed daily cost for facility $j$ |
| $C_j$ | capacity of facility $j$ |
| $M_j$ | big-M value for facility $j$ (typically $M_j = C_j$) |


<br>The objective is to minimize the total daily system cost:

$$
Z = C_{trans} + C_{tip} + C_{rec} + C_{fixed}.
$$

where 
- transport cost from cities to facilities:

$$
C_{trans,city} = t \sum_{i=1}^3 \sum_{j\in\{LF,MRF,WTE\}} d_{i,j}\,x_{i,j}.
$$

- transport cost for inter-facility flows:

$$
C_{trans,inter} = t\big(d_{MRF,LF}\,y_{MRF,LF} + d_{MRF,WTE}\,y_{MRF,WTE} + d_{WTE,LF}\,y_{WTE,LF}\big).
$$

1. Total transport cost

$$
C_{trans} = C_{trans,city} + C_{trans,inter}.
$$

2. Tipping fees

$$
C_{tip}
= c^{tip}_{LF}\big(\sum_{i=1}^3 x_{i,LF} + y_{MRF,LF} + y_{WTE,LF}\big)
+ c^{tip}_{MRF}\big(\sum_{i=1}^3 x_{i,MRF}\big)
+ c^{tip}_{WTE}\big(\sum_{i=1}^3 x_{i,WTE} + y_{MRF,WTE}\big).
$$


3. MRF recycling costs

$$
C_{rec} = c^{rec}\, r_{\text{MRF}} \big(\sum_{i=1}^3 x_{i,MRF}\big).
$$

4. Fixed facility operating costs

$$
C_{fixed} = F_{LF}\,\delta_{LF} + F_{MRF}\,\delta_{MRF} + F_{WTE}\,\delta_{WTE}.
$$

Putting all components together:

$$
Z = C_{trans} + C_{tip} + C_{rec} + C_{fixed}.
$$

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

**Answer**:

City mass balance (for $i = 1,2,3$): 
$$
\sum_{j\in\{LF,MRF,WTE\}} x_{i,j} = S_i.
$$

MRF inflow and split: 
$$
X_{MRF} = \sum_{i=1}^3 x_{i,MRF}.
$$ 

$$
M^{rec} = r_{\text{MRF}}\,X_{MRF}, \quad M^{res} = (1-r_{\text{MRF}})\,X_{MRF}.
$$

MRF residual routing: 
$$
y_{MRF,LF} + y_{MRF,WTE} = (1-r_{\text{MRF}})\sum_{i=1}^3 x_{i,MRF}.
$$

WTE inflow: 
$$
X_{WTE} = \sum_{i=1}^3 x_{i,WTE} + y_{MRF,WTE}.
$$

WTE ash: 
$$
y_{WTE,LF} = 0.1641\sum_{i=1}^3 x_{i,WTE} + 0.1386\,y_{MRF,WTE}.
$$

Landfill inflow: 
$$
X_{LF} = \sum_{i=1}^3 x_{i,LF} + y_{MRF,LF} + y_{WTE,LF}.
$$

Capacity constraints: 
$$ 
\sum_{i=1}^3 x_{i,MRF} \le C_{MRF},
$$ 

$$
\sum_{i=1}^3 x_{i,WTE} + y_{MRF,WTE} \le C_{WTE},
$$ 
$$
\sum_{i=1}^3 x_{i,LF} + y_{MRF,LF} + y_{WTE,LF} \le C_{LF}.
$$

Big-M linking constraints:

(to enforce zero flow when facility closed)
$$
\sum_{i=1}^3 x_{i,MRF} \le M_{MRF}\,\delta_{MRF},
$$ 

$$
\sum_{i=1}^3 x_{i,WTE} + y_{MRF,WTE} \le M_{WTE}\,\delta_{WTE},
$$ 

$$
\sum_{i=1}^3 x_{i,LF} + y_{MRF,LF} + y_{WTE,LF} \le M_{LF}\,\delta_{LF}.
$$ 
- Choose $M_j\ge C_j$, typically $M_j=C_j$.

Nonnegativity and integrality: 
$$
x_{i,j} \ge 0,\; y_{MRF,LF}\ge0,\; y_{MRF,WTE}\ge0,\; y_{WTE,LF}\ge0.
$$ 

$$
\delta_{LF},\delta_{MRF},\delta_{WTE} \in \{0,1\}.
$$

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [8]:
# Data
S = [100.0, 90.0, 120.0]
C_LF = 200.0; C_MRF = 350.0; C_WTE = 210.0

# MRF recycling rate = 40%
r_MRF = 0.40

# Ash fractions from problem 1.1
a_MSW = 0.1641  
a_res = 0.1386 

# Transportation cost ($/Mg-km)
t = 1.5

# Distances (km)
d_city_LF = [5.0, 15.0, 13.0]
d_city_MRF = [30.0, 25.0, 45.0]
d_city_WTE = [15.0, 10.0, 20.0]
d_MRF_LF = 32.0; d_MRF_WTE = 15.0; d_WTE_LF = 18.0

# Tipping fees ($/Mg)
c_tip_LF = 50.0; c_tip_MRF = 7.0; c_tip_WTE = 60.0

# Recycling cost ($/Mg recycled)
c_rec = 40.0

# Fixed costs ($/day)
F_LF = 2000.0; F_MRF = 1500.0; F_WTE = 2500.0

# Big-M values (choose tight = capacities)
M_LF = C_LF; M_MRF = C_MRF; M_WTE = C_WTE

# Model
model = Model(HiGHS.Optimizer)
set_silent(model) 

# Variables
@variable(model, x[i = 1:3, j = 1:3] >= 0) 
@variable(model, y_MRF_LF >= 0)   
@variable(model, y_MRF_WTE >= 0)   
@variable(model, y_WTE_LF >= 0)    
@variable(model, delta_LF, Bin)    
@variable(model, delta_MRF, Bin)   
@variable(model, delta_WTE, Bin)  

# City mass balance constraints
@constraint(model, [i = 1:3], sum(x[i, j] for j = 1:3) == S[i])

# MRF residue split for non-recycled portion
@constraint(model, y_MRF_LF + y_MRF_WTE == (1 - r_MRF) * sum(x[i, 2] for i = 1:3))

# WTE ash
@constraint(model, y_WTE_LF == a_MSW * sum(x[i, 3] for i = 1:3) + a_res * y_MRF_WTE)

# Capacity constraints
@constraint(model, sum(x[i, 1] for i = 1:3) + y_MRF_LF + y_WTE_LF <= C_LF)
@constraint(model, sum(x[i, 2] for i = 1:3) <= C_MRF)
@constraint(model, sum(x[i, 3] for i = 1:3) + y_MRF_WTE <= C_WTE)

# Big-M linking constraints
@constraint(model, sum(x[i, 1] for i = 1:3) + y_MRF_LF + y_WTE_LF <= M_LF * delta_LF)
@constraint(model, sum(x[i, 2] for i = 1:3) <= M_MRF * delta_MRF)
@constraint(model, sum(x[i, 3] for i = 1:3) + y_MRF_WTE <= M_WTE * delta_WTE)

# Objective: Minimize total cost
transport_city = t * (
    sum(d_city_LF[i] * x[i, 1] for i = 1:3) + 
    sum(d_city_MRF[i] * x[i, 2] for i = 1:3) +
    sum(d_city_WTE[i] * x[i, 3] for i = 1:3)
)

transport_inter = t * (d_MRF_LF * y_MRF_LF + d_MRF_WTE * y_MRF_WTE + d_WTE_LF * y_WTE_LF)

tip_cost = c_tip_LF * (sum(x[i, 1] for i = 1:3) + y_MRF_LF + y_WTE_LF) +
           c_tip_MRF * (sum(x[i, 2] for i = 1:3)) +
           c_tip_WTE * (sum(x[i, 3] for i = 1:3) + y_MRF_WTE)

recycle_cost = c_rec * r_MRF * sum(x[i, 2] for i = 1:3)

fixed_cost = F_LF * delta_LF + F_MRF * delta_MRF + F_WTE * delta_WTE

@objective(model, Min, transport_city + transport_inter + tip_cost + recycle_cost + fixed_cost)

# Solve
optimize!(model)

# Output results
status = termination_status(model)
println("Status: ", status)

if status == OPTIMAL
    println("Objective value: \$", round(objective_value(model); digits = 2))

    println("\nFlows (x[i, j]) where j = 1:LF, 2:MRF, 3:WTE:")
    for i in 1:3, j in 1:3
        v = value(x[i, j])
        if v > 1e-6
            println("x[$i, $j] = ", round(v; digits = 3))
        end
    end

    println("\ny_MRF_LF = ", round(value(y_MRF_LF); digits = 3))
    println("y_MRF_WTE = ", round(value(y_MRF_WTE); digits = 3))
    println("y_WTE_LF = ", round(value(y_WTE_LF); digits = 3))

    println("\ndelta_LF  = ", Int(value(delta_LF)))
    println("delta_MRF = ", Int(value(delta_MRF)))
    println("delta_WTE = ", Int(value(delta_WTE)))
end

Status: OPTIMAL
Objective value: $27855.48

Flows (x[i, j]) where j = 1:LF, 2:MRF, 3:WTE:
x[1, 1] = 100.0
x[2, 3] = 90.0
x[3, 1] = 78.405
x[3, 3] = 41.595

y_MRF_LF = 0.0
y_MRF_WTE = 0.0
y_WTE_LF = 21.595

delta_LF  = 1
delta_MRF = 0
delta_WTE = 1


**Answer**:

The optimal solution is WTE takes all the 90 Mg of wastes from City 2, then another 41.595 Mg from City 3. It will generate about 21.595 Mg ashes that will then be transported to LF. More than the ashes, LF also takes 100 Mg of wastes from City 1 and 78.405 Mg of wastes from City 3. This gives MTE 131.595 Mg of total wastes to treat, and 200 Mg afterwards in total for LF, while MRF is completely unused. The optimal objective value is $27855.48.

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

**Answer**:

![Description](problem1.6.png)

The facility that will not be used is the MRF. This solution makes sense because using the MRF incurs not only a fixed daily cost of $1500, but also a processing cost of $40/Mg recylced. With a low tipping fee of $7/Mg, the savings from sending waste through MRF versus other alternatives is small, so the combined costs outweigh the savings here. In addition, with the LF capacity available up to 200 Mg and WTE available up to 210 Mg, the model can meet demand without paying the MRF costs. Thus, not using MRF is economically and physically consistent given the data, with capacity limits satisfied, mass conserved, and objective minimized. 

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

**Answer**:

![Description](problem2.1.png)

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

**Answer**:

Sets:
- Generators $i \in \mathcal{G}$.
- Time periods $t = 1,2$.
- Scenarios $s \in \mathcal{S} = \{1, 2, 3, 4\}$.

Parameters:
- $P_i^{\min}, P_i^{\max}$ = min/max output of generator $i$.
- $c_i$ = variable cost of generator $i$.
- $R_i$ = ramp limit of generator $i$.
- $d_1 = 1100$.
- $d_2^s \in \{1200, 1500\}$ depending on scenario.
- Solar/wind capacity factors:
  - Period 1: $\text{CF}^{(1)}_{\text{solar}} = 0.9,\ \text{CF}^{(1)}_{\text{wind}} = 0.45$.
  - Period 2: $\text{CF}^s_{\text{solar}},\ \text{CF}^s_{\text{wind}}$.
- Scenario probabilities:
  - $p_1 = 0.525,\ p_2 = 0.225,\ p_3 = 0.175,\ p_4 = 0.075$.

Decision variables:
- $g_{i,1}$ = generation of unit $i$ in period 1.
- $g_{i,2}^s$ = generation of unit $i$ in period 2 under scenario $s$.

Objective:
$$
\min\ \sum_{i} c_i g_{i,1}
\ +\ \sum_{s} p_s \sum_{i} c_i g_{i,2}^s
$$

Constraints:

1. Period-1 capacity limits:
$$
P_i^{\min} \le g_{i,1} \le P_i^{\max}.
$$

2. Period-1 renewable availability:
$$
g_{\text{wind},1} \le 0.45 \, P_{\text{wind}}^{\max},
\qquad
g_{\text{solar},1} \le 0.90 \, P_{\text{solar}}^{\max}.
$$

3. Period-1 demand:
$$
\sum_i g_{i,1} = 1100.
$$

4. Period-2 capacity limits:
$$
P_i^{\min} \le g_{i,2}^s \le P_i^{\max} \qquad \forall s.
$$

5. Period-2 renewable availability:
$$
g_{\text{wind},2}^s \le \text{CF}^s_{\text{wind}}\, P_{\text{wind}}^{\max},
\qquad
g_{\text{solar},2}^s \le \text{CF}^s_{\text{solar}}\, P_{\text{solar}}^{\max}
\qquad \forall s.
$$

6. Period-2 demand (scenario-dependent):
$$
\sum_i g_{i,2}^s = d_2^s \qquad \forall s.
$$

7. Ramping:
$$
-R_i \le g_{i,2}^s - g_{i,1} \le R_i \qquad \forall i,s.
$$

8. Nonnegativity:
$$
g_{i,1} \ge 0,\qquad g_{i,2}^s \ge 0.
$$

## References

List any external references consulted, including classmates.

Used ChatGPT for problems 1.3, 1.4, 2.2, for the purpose of helping with LaTex syntax, since it is really complicated to utilize with all the different symbols and versions. OpenAI. (2025). ChatGPT (Feb 1 GPT-4 model) [Large language model]. https://chat.openai.com